# Module 2: Tracing

> ~25 min

With tracing on in `.env` (Module 0), LangSmith records every agent run as a **trace**: a tree of **runs**, one per model call, tool call and subagent step, each with its inputs, outputs, errors and latency. You get this without instrumenting any code, and this agent's traces land in its **development** environment. In this module you generate a batch of realistic conversations, turn on **Engine** so it starts scanning them, and then explore the same traces yourself: one thread in the **Trajectory** view, the search bar, AI search, and queries from the SDK, the CLI and your coding agent. By the end you will have found, by hand, a failure pattern that Engine reports on in Module 3.

## Setup

This cell builds the agent and looks up the tracing project behind the agent's `development` environment. The SDK and CLI queries below use that project's ID. The cell prints the agent and a link to its **Tracing** page.

> **Project fallback:** on self-hosted or older orgs without agents, traces go to `LANGSMITH_PROJECT` (default `langsmith-poc`), and this cell resolves that project instead.

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from collections import Counter
from datetime import datetime, timedelta, timezone

from langsmith import Client, uuid7

from agents.client_research_agent import build_client_research_agent
from utils.config import AGENT_ENVIRONMENT, AGENT_NAME, TRACING_MODE, environment_project
from utils.notebook import trace_link
from utils.traffic import generate_traffic

client = Client()
agent = build_client_research_agent()
dest = environment_project(client)
project_id = dest.project_id
since = datetime.now(timezone.utc) - timedelta(days=7)  # SDK queries default to the last 24h

print("Agent:", AGENT_NAME, "/", AGENT_ENVIRONMENT, f"({TRACING_MODE} mode)")
trace_link(dest.url, "Tracing")

## 1. Generate traffic

`generate_traffic()` replays 24 scripted conversations through the agent, five at a time, and tags each turn with `source: tour-seed`. Some threads have one turn and some have several; the mix covers client lookups, market research and a few out-of-scope asks. That gives search, Engine and the later modules realistic data instead of a handful of hand-typed runs. The run takes about 4 minutes and prints how many threads finished, plus a link to the project.

In [ ]:
thread_ids = generate_traffic()

print("Threads generated:", len(thread_ids))
trace_link(dest.url, "Threads")

Open the link, then the **Threads** tab. Each row is one conversation, and the turn count shows which ones went past a single question.

<img src="../images/warm_up_traces.png" alt="Tracing table with the Threads, Traces and Runs tabs" style="width: auto; max-height: 360px; border-radius: 8px;">

## 2. Turn on Engine

**Engine** reads an environment's traces on a schedule, groups recurring failures into **issues**, and proposes fixes, so you don't read traces one by one to spot a failure that repeats across conversations. It needs traces from the last 7 days before it can start, which is why traffic came first.

> **Requires:** Engine turned on for your organization: an org admin clicks **Enable Engine for this Organization** on the **Engine** page, and anyone else sees **Request access from an admin**. **Without it:** skip this section; Module 3 opens with a fallback.

1. Open **Engine** in the sidebar. Under **Choose an environment to analyze**, pick **Development** for `client-research-agent`. If Engine already analyzes another project, open the project selector and choose **Set up another project** first.
2. Optional: under **Connect your agent's code repository**, pick your fork of this repo. Leave **Open a pull request for every fix** off.
3. Under **What matters most to you?**, select **Tool Call Failures** and **Hallucinations**. Click **Custom Category**, paste the priority printed by the next cell, and click **Add**.
4. Leave **Analysis level** and **Focus on specific traces** as they are, and click **Start Analyzing**.

> **Requires:** a fork of this repo with the LangSmith GitHub App installed. **Without it:** skip step 2; Engine still files issues from traces, and Module 3 skips the pull request step.

<img src="../images/engine_config.png" alt="Engine setup panel, Find and fix your agent's issues" style="width: auto; max-height: 360px; border-radius: 8px;">

In [ ]:
ENGINE_PRIORITY = (
    "Client lookups that fail repeatedly: get_client_profile is called several "
    "times in one conversation and never returns a profile."
)

print("Custom priority:", ENGINE_PRIORITY)
trace_link(dest.url, "Tracing (Engine is in the sidebar)")

The first scan takes about 20 minutes. Keep going: the rest of this module covers that wait, and Module 3 opens with what Engine found.

## 3. A multi-turn thread

A **thread** is a series of traces that share a `thread_id` (or `session_id` / `conversation_id`) in their metadata, so you can read a whole conversation as one unit instead of hunting for its traces. LangGraph copies `thread_id` from the run config into that metadata, and the agent's checkpointer uses the same ID to remember earlier turns.

The first cell sets two questions and one thread ID. The second sends them as two `invoke` calls with that ID and prints each answer; it takes about a minute.

In [ ]:
TURNS = [
    "Pull up Acme Pension Fund and summarize their top holdings.",
    # Naming the subagent makes the delegation show up in section 4
    "Have the research agent give me a one-paragraph update on AAPL. Search at most once.",
]

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}  # same config on every call = one thread

print("Thread ID:", thread_id)

In [ ]:
for question in TURNS:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
    print("Answer:", result["messages"][-1].text[:200], "\n")

print("Turns sent:", len(TURNS))
trace_link(dest.url, "Threads")

In **Threads**, open the row with this thread ID and switch the side panel to **Turns**. Both questions appear as cards in one conversation.

## 4. Read the trajectory

The **Trajectory** view shows a thread as the conversation the model saw: each turn's answer, the tool calls it made with their results, and subagent work inline, so you follow the agent's reasoning without opening nested runs one by one.

In the same thread, switch to **Trajectory** (press `T`). Expand the grouped `get_client_profile` row, then click into the `research-agent` block to see the subagent's own messages.

Count the `get_client_profile` calls in the first turn. The agent passes a slug built from the client's name, gets an error back, and usually tries again with a different value before it answers that it can't find the client. Click one failed call to open it in **Details** (press `D`) with its error and inputs.

<img src="../images/TODO-02-trajectory-failed-lookup.png" alt="Trajectory view with repeated failed get_client_profile calls, one open in Details" style="width: auto; max-height: 360px; border-radius: 8px;">

## 5. Search with the search bar

The search bar filters the table on run fields such as name, type, status, latency, tags and metadata, with clauses such as `run_type:tool AND status:error`. The scope selector to its left sets which runs a clause must match: the **Thread**, the **Root run**, a **Single run**, or **Any run** in the tree.

Switch to **Runs** and search `name:get_client_profile AND status:error`.

> The search bar syntax is not the SDK filter syntax in section 7.

<img src="../images/filter_traces.png" alt="Tracing search bar filtering runs" style="width: auto; max-height: 360px; border-radius: 8px;">

Switch to **Threads**, keep the scope at **Any run**, and search `run_type:tool AND status:error`. Then click **Add filter scope**, set the new row to **Thread**, and search `turn_count:>1`. The table now holds only multi-turn conversations with a failed tool call.

<img src="../images/TODO-02-search-scopes.png" alt="Threads search with an Any run row and a Thread row" style="width: auto; max-height: 360px; border-radius: 8px;">

## 6. Search with AI

**AI search** matches threads, traces or runs by a plain-language description of what happened in the conversation, for questions a field filter can't express.

> **Requires:** AI search (beta, LangSmith Cloud), turned on by an org admin under **Settings** → **AI search** → **Enable AI search**; it processes trace data in the US. **Without it:** skip to section 7, which finds the same threads with a filter.

In **Threads**, open the scope selector, choose **AI search**, and enter `the agent could not find the client the user asked about`. Results show a **Confidence** column, and opening one highlights the matching messages in **Trajectory**.

<img src="../images/TODO-02-ai-search.png" alt="AI search row with the Confidence column" style="width: auto; max-height: 360px; border-radius: 8px;">

The next cell runs the same AI search from the SDK with `ai_search=` and prints how many threads matched, plus the first question of three. It raises an error if AI search is off for your org.

In [ ]:
AI_QUERY = "the agent could not find the client the user asked about"

matched = []
async for thread in client.threads.streaming.query_items(
    project_id=project_id,
    ai_search=AI_QUERY,
    min_start_time=since,  # defaults to the last 24h
):
    matched.append(thread)

print("Threads matched by AI search:", len(matched))
for thread in matched[:3]:
    print("-", (thread.first_inputs or "")[:80])

The matches include client lookups whose final answer never errored, so a `status:error` filter on the root run would miss them.

## 7. Query traces from the SDK

`client.runs.query()` and `client.threads.query()` take the same filters as code, so a script or CI job can count and group failures across every trace without exporting them. Both are async iterators, take a project **ID**, and only look back 24 hours unless you pass `min_start_time`.

The next cell pulls every failed `get_client_profile` run from the last 7 days, counts the threads with two or more, and prints a few inputs with their errors.

In [ ]:
failed = []
async for run in client.runs.query(
    project_ids=[project_id],
    run_type="TOOL",
    has_error=True,
    filter='eq(name, "get_client_profile")',
    min_start_time=since,
    selects=["THREAD_ID", "INPUTS", "ERROR"],  # fetch only the fields used below
):
    failed.append(run)

# A thread with several failed lookups is one conversation stuck guessing
per_thread = Counter(run.thread_id for run in failed)
repeat_threads = [tid for tid, count in per_thread.items() if count >= 2]

print("Failed lookups:", len(failed))
print("Threads with 2+ failed lookups:", len(repeat_threads))
for run in failed[:5]:
    print(run.inputs, "->", (run.error or "")[:80])

Several threads hold two or more failed lookups, each with a different `client_id`. That repetition across conversations is the pattern Engine is scanning for.

The next cell asks the same question per conversation: `client.threads.query()` with a `tree_filter` returns each thread that holds a failed lookup, and prints its turn count and first question.

In [ ]:
TREE_FILTER = 'and(eq(name, "get_client_profile"), eq(status, "error"))'

threads = []
async for thread in client.threads.query(
    project_id=project_id,
    tree_filter=TREE_FILTER,
    min_start_time=since,
):
    threads.append(thread)

print("Threads with a failed lookup:", len(threads))  # 7-day window, so earlier runs count too
for thread in threads[:3]:
    print(f"- {thread.count} turn(s):", (thread.first_inputs or "")[:80])

`tree_filter` matches a thread if any run in any of its traces matches: the SDK form of the **Any run** scope from section 5.

## 8. Query from the CLI and your coding agent

The `langsmith` CLI and the LangSmith Remote MCP server expose the same runs and threads to a terminal or a coding agent, using the setup from Module 0, so a coding agent can debug from real traces without you pasting them in. Inside this notebook, `!langsmith` reads `LANGSMITH_API_KEY` from your `.env`.

The next cell lists the five most recent failed `get_client_profile` runs from the last 4 hours, with their inputs and errors.

In [ ]:
!langsmith runs list --project-id {project_id} --run-type tool --name get_client_profile --error --include-io --last-n-minutes 240 --limit 5

Your coding agent asks the same question through the Remote MCP server's `query_runs` tool. The next cell prints a prompt; paste it into Claude Code or Cursor.

> **Requires:** the Remote MCP server added to your coding agent in Module 0. **Without it:** skip this step; the CLI table above has the same answer.

In [ ]:
MCP_PROMPT = (
    "Using the langsmith MCP server, query the failed get_client_profile tool runs "
    f"from the last 4 hours in project {dest.project_name}. "
    "List the client_id values the agent tried and the error each one returned."
)

print(MCP_PROMPT)

The agent's answer lists the same `client_id` guesses as the SDK query in section 7, read straight from your traces.

## 9. Check on Engine

Engine's first scan finishes about 20 minutes after you click **Start Analyzing**, so it may be done by now. Open **Engine**; if it asks you to review an **agent overview**, read it, correct anything wrong, and continue. The next cell lists the issues Engine has filed for this project from the CLI.

In [ ]:
!langsmith projects issues list --project-id {project_id} --format pretty

An empty list means the scan is still running. Module 3 starts there either way.

## Recap

| Step | Where it lives |
|---|---|
| Generate traffic | `generate_traffic()` → agent **Development** environment |
| Turn on Engine | **Engine** → **Start Analyzing** |
| Multi-turn thread | same `thread_id` in `configurable` |
| Read a thread | **Trajectory** / **Turns** / **Details** |
| Search | search bar + scopes **Thread**, **Root run**, **Single run**, **Any run** |
| AI search | scope **AI search** · `client.threads.streaming.query_items(ai_search=...)` |
| SDK | `client.runs.query()`, `client.threads.query()` |
| CLI | `langsmith runs list`, `langsmith projects issues list` |
| Coding agent | Remote MCP `query_runs` |

**Next:** Module 3 — Engine

**Docs:** [View traces](https://docs.langchain.com/langsmith/view-traces) · [Filter traces](https://docs.langchain.com/langsmith/filter-traces) · [Query threads](https://docs.langchain.com/langsmith/query-threads) · [Query runs](https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs) · [LangSmith CLI](https://docs.langchain.com/langsmith/langsmith-cli) · [Remote MCP](https://docs.langchain.com/langsmith/langsmith-remote-mcp) · [Set up Engine](https://docs.langchain.com/langsmith/engine-setup)